In [ ]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [3]:
"D:\current documents\Desktop\Final project\Radiography\test"
"D:\current documents\Desktop\Final project\Radiography\train"
"D:\current documents\Desktop\Final project\Radiography\val"


'D:\\current documents\\Desktop\\Final project\\Radiography\x0bal'

In [4]:
test_data="D:\current documents\Desktop\Final project\Radiography\test"
train_data="D:\current documents\Desktop\Final project\Radiography\train"
val_data="D:\current documents\Desktop\Final project\Radiography\val"


In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_gen = ImageDataGenerator(rescale=1./255)

# Train
train_data = train_gen.flow_from_directory(
    r"D:\current documents\Desktop\Final project\Radiography\train",
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

# Validation
val_data = test_gen.flow_from_directory(
    r"D:\current documents\Desktop\Final project\Radiography\val",
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

# Test
test_data = test_gen.flow_from_directory(
    r"D:\current documents\Desktop\Final project\Radiography\test",
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 485 images belonging to 2 classes.


In [ ]:
#build a cnnmodel
model = Sequential()

# Convolution Layer 1
model.add(Conv2D(32, (3,3), activation='relu',
                 input_shape=(224,224,3)))
model.add(MaxPooling2D(pool_size=(2,2)))

# Convolution Layer 2
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Convolution Layer 3
model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Flatten
model.add(Flatten())

# Fully Connected Layers
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

# Output Layer
model.add(Dense(1, activation='sigmoid'))


#compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
#train it
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)
model.save("cnn_model.h5")

c:\Users\admin\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 370s 2s/step - accuracy: 0.8273 - loss: 0.4143 - val_accuracy: 0.6250 - val_loss: 0.9277
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 229s 1s/step - accuracy: 0.8846 - loss: 0.2746 - val_accuracy: 0.8125 - val_loss: 0.5964
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 200s 1s/step - accuracy: 0.8944 - loss: 0.2597 - val_accuracy: 0.8750 - val_loss: 0.6362
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 200s 1s/step - accuracy: 0.9028 - loss: 0.2379 - val_accuracy: 0.8750 - val_loss: 0.4781
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 200s 1s/step - accuracy: 0.9176 - loss: 0.2151 - val_accuracy: 0.6250 - val_loss: 0.7177
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 25097s 155s/step - accuracy: 0.9199 - loss: 0.2032 - val_accuracy: 0.6250 - val_loss: 0.8951
Epoch 7/10
132/163 ━━━━━━━━━━━━━━━━━━━━ 4:50 9s/step - accuracy: 0.9320 - loss: 0.1886

loss, accuracy = model.evaluate(test_data)
print("Test Accuracy:", accuracy)
print("Test Loss:", loss)

# model
model.save("cnn_model.h5")

In [12]:
print(train_data.class_indices)

{'NORMAL': 0, 'PNEUMONIA': 1}


In [13]:
prediction = model.predict(img_array)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
[[0.62914705]]


In [16]:
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
import numpy as np

model = load_model("cnn_model.h5")

image_path = r"D:\current documents\Desktop\Final project\Radiography\val\NORMAL\NORMAL2-IM-1427-0001.jpeg"

img = image.load_img(image_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = img_array / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array, verbose=0)
score = prediction[0][0]

print("Raw prediction:", score)

if score > 0.5:
    print(f"Prediction: PNEUMONIA ({score*100:.2f}%)")
else:
    print(f"Prediction: NORMAL ({(1-score)*100:.2f}%)")

Raw prediction: 0.62914705
Prediction: PNEUMONIA (62.91%)
